# Catalogues, selection and pair finding

This section discusses the catalog fields that are taken from each simulation and how these fields are used in the pipeline.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / 'pyproject.toml').is_file())
def table(relative):
    return pd.read_csv(ROOT / 'results' / relative)
def saved_json(relative):
    return json.loads((ROOT / 'results' / relative).read_text(encoding='utf-8'))
plt.rcParams.update({'figure.figsize': (9, 4), 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_rows', 40)
pd.set_option('display.max_columns', 16)

We compare FLAMINGO L1_m8 and TNG300-1 at z=0. Positions are physical kpc,
velocities are km/s, and masses are solar masses. FLAMINGO catalogue masses multiply
by 10^10; native TNG masses multiply by 10^10/h, with h=0.6774.

Galaxy eligibility uses 2×10^10 ≤ Mstar ≤ 5×10^11. FLAMINGO also requires at least
100 stellar particles; the TNG quality cut uses SubhaloFlag == 1. The selected host representation
contains its eligible central and most massive eligible satellite. The pair finder uses
periodic distances and 500–1000 kpc separation. Peculiar relative velocities satisfy
-400 ≤ vr ≤ 0, 0 ≤ vt ≤ 300, and total speed ≤500 km/s.

The two isolation tests use different masses. The host third-member veto uses stellar
mass ≥1.5 times the larger pair stellar mass. The 2000 kpc intruder veto uses total
bound mass >0.5 times the summed pair bound mass around the periodic mass-weighted
pair centre. Read the implementations for the precise search populations.


### Galaxy selection and initial candidates

The next two tables are transcribed from thesis Tables 3.2–3.3, rather than recomputed
from the parent catalogues in this notebook. They preserve the original selection-stage
counts. The later cutflow and final-pair statistics describe different stages.

Eligible satellites count qualifying satellite candidates. The final representation uses
only the most massive eligible satellite per host, so the number of selected satellites
is the number of groups with satellites. Initial colour classes use g-r < 0.68 for blue
and g-r ≥ 0.68 for red.


In [2]:
selection = pd.DataFrame({
    'quantity': ['h','box_side_physical_kpc','raw_subhaloes','after_quality_selection',
                 'after_stellar_mass_window','eligible_centrals','eligible_satellites',
                 'final_sample_galaxies','groups_with_satellites','groups_with_central_only'],
    'flamingo': [.681,1_000_000,81_415_840,6_519_590,4_687_523,2_919_928,824_080,
                 3_366_610,446_682,2_473_246],
    'tng': [.6774,302_627.69,14_485_709,1_902_558,60_562,38_778,12_912,
            45_392,6_614,32_164]})
initial_candidates = pd.DataFrame({
    'quantity': ['pairs','same_host','different_host','blue_blue','red_red','mixed_blue_red',
                 'mean_separation_kpc','mean_relative_speed_km_s','mean_bound_mass_ratio'],
    'flamingo': [296_516,102_219,194_297,62_096,93_765,140_655,775.14,230.99,.455],
    'tng': [2_655,2_084,571,288,1_265,1_102,746.42,269.33,.331]})
display(selection.set_index('quantity'))
display(initial_candidates.set_index('quantity'))
assert 2_919_928 + 446_682 == 3_366_610
assert 38_778 + 6_614 == 45_392
assert 102_219 + 194_297 == 296_516
assert 2_084 + 571 == 2_655

,flamingo,tng
quantity,,
h,6.810000e-01,6.774000e-01
box_side_physical_kpc,1.000000e+06,3.026277e+05
raw_subhaloes,8.141584e+07,1.448571e+07
after_quality_selection,6.519590e+06,1.902558e+06
after_stellar_mass_window,4.687523e+06,6.056200e+04
eligible_centrals,2.919928e+06,3.877800e+04
eligible_satellites,8.240800e+05,1.291200e+04
final_sample_galaxies,3.366610e+06,4.539200e+04
groups_with_satellites,4.466820e+05,6.614000e+03


,flamingo,tng
quantity,,
pairs,296516.000,2655.000
same_host,102219.000,2084.000
different_host,194297.000,571.000
blue_blue,62096.000,288.000
red_red,93765.000,1265.000
mixed_blue_red,140655.000,1102.000
mean_separation_kpc,775.140,746.420
mean_relative_speed_km_s,230.990,269.330
mean_bound_mass_ratio,0.455,0.331


In [3]:
cutflow = pd.DataFrame({
    'stage': ['candidate pairs','after host third-member veto','after intruder veto','after kinematics'],
    'flamingo': [296516,275878,182193,142713], 'tng': [2655,2601,2235,1530]})
display(cutflow)
audited = saved_json('populations/calculated_tables.json')
display(pd.DataFrame(audited['densities']).T)
display(pd.DataFrame(audited['pair_stats']).T)
density = pd.DataFrame(audited['densities']).T
assert np.isclose(density.loc['tng','volume'], (205 / .6774)**3)
print('FLAMINGO / TNG final pair number density:',
      density.loc['flamingo','pair_number_density'] / density.loc['tng','pair_number_density'])

FLAMINGO / TNG final pair number density: 2.5852235952200955


,stage,flamingo,tng
0,candidate pairs,296516,2655
1,after host third-member veto,275878,2601
2,after intruder veto,182193,2235
3,after kinematics,142713,1530


,volume,side,stellar_selected_number_density,sample_number_density,pair_number_density,pair_efficiency
flamingo,1.000000e+09,1000.000000,0.004688,0.003367,0.000143,0.042391
tng,2.771571e+07,302.627694,0.002185,0.001638,0.000055,0.033706


,n,same,different,blue_blue,red_red,mixed,mean_separation,mean_velocity,mean_ratio
flamingo,142713.0,63071.0,79642.0,25465.0,49609.0,67639.0,764.872352,177.661790,0.423803
tng,1530.0,1253.0,277.0,157.0,752.0,621.0,747.336914,221.866119,0.340387


### The target changes with host membership

Same-host pairs use a single host's M200c, whilst different-host pairs sum the
independent M200c values. The alternative target sums the two total bound subhalo
masses in both classes. These mass definitions are not interchangeable.

The real MW–M31 system is different-host. Pooled feature support and pooled accuracy
do not by themselves validate an exclusively two-host inference.

The physical TNG box volume is 2.771571×10^7 Mpc³. This follows from the box side
205/h Mpc, with h=0.6774, and is the volume used for the number densities above.

### Catalogue code

The [FLAMINGO pipeline](../research/analogues/flamingo_lg_pipeline.py),
[TNG pair pipeline](../research/TNG300/analogues/pipeline.py), and
[selection configuration](../research/analogues/flamingo_selection_config.json)
contain the full selection procedure. [Notebook 90](90_generate_catalogue_caches.ipynb)
saves the sample/pair dictionaries and corrected arrays. Use that notebook for a new
catalogue build: the TNG migration script expects an existing set of caches.


[Previous](00_start_here.ipynb) · [Next](02_features_and_population_shift.ipynb) · [Repository map](../README.md)
